# 02 — Feature Engineering

## Objective

Transform the cleaned credit-card data into financially meaningful
features that can help predict customer default risk.

The engineered features focus on:

- repayment behavior
- credit utilization
- payment behavior

The raw variables are retained so that the final model can use both
original and engineered information.

In [21]:
import pandas as pd
import numpy as np

In [22]:
clean_path = "../data/processed/credit_risk_clean.csv"

df = pd.read_csv(clean_path)

print(f"Dataset shape: {df.shape}")

Dataset shape: (29965, 24)


## 2. Repayment Behavior Features

The dataset contains repayment-status information for six months.

We create summary features that capture:

- how many months the customer experienced a payment delay
- the customer's worst observed delay
- the average severity of actual delays
- the most recent observed delay

These features provide a more compact representation of repayment behavior.

In [23]:
repayment_cols = [
    "repayment_sep",
    "repayment_aug",
    "repayment_july",
    "repayment_jun",
    "repayment_may",
    "repayment_apr"
]

### Feature 1 - Number of delayed months

In [24]:
df["num_delayed_months"] = (
    df[repayment_cols] > 0
).sum(axis=1)

### Feature 2 - Maximum delay

In [25]:
df["max_delay"] = (
    df[repayment_cols]
    .clip(lower=0)
    .max(axis=1)
)

### Feature 3 - Average delay

In [26]:
delay_values = df[repayment_cols].where(
    df[repayment_cols] > 0
)

df["avg_delay"] = (
    delay_values.mean(axis=1)
    .fillna(0)
)

### Feature 4 - Most recent delay

In [27]:
df["recent_delay"] = (
    df["repayment_sep"].clip(lower=0)
)

### Validation

We verify the ranges and distributions of the engineered repayment
features before using them in the model.

In [28]:
repayment_features = [
    "num_delayed_months",
    "max_delay",
    "avg_delay",
    "recent_delay"
]

df[repayment_features].describe().T

,count,mean,std,min,25%,50%,75%,max
num_delayed_months,29965.0,0.834273,1.554821,0.0,0.0,0.0,1.000000,6.0
max_delay,29965.0,0.682163,1.073847,0.0,0.0,0.0,2.000000,8.0
avg_delay,29965.0,0.621311,0.931530,0.0,0.0,0.0,1.666667,6.0
recent_delay,29965.0,0.356416,0.760752,0.0,0.0,0.0,0.000000,8.0


In [29]:
df[repayment_features].isnull().sum()

num_delayed_months    0
max_delay             0
avg_delay             0
recent_delay          0
dtype: int64

In [30]:
df[
    repayment_features
].head(10)

,num_delayed_months,max_delay,avg_delay,recent_delay
0,2,2,2.0,2
1,2,2,2.0,0
2,0,0,0.0,0
3,0,0,0.0,0
4,0,0,0.0,0
5,0,0,0.0,0
6,0,0,0.0,0
7,0,0,0.0,0
8,1,2,2.0,0
9,0,0,0.0,0


## 3. Credit Utilization Features

Credit utilization measures the customer's statement balance relative
to their available credit limit.

We create:

- September utilization
- average six-month utilization
- maximum six-month utilization
- average six-month bill utilization

Negative statement amounts are clipped to zero only for these derived
utilization calculations. The original bill variables remain unchanged.

In [31]:
bill_cols = [
    "bill_sep",
    "bill_aug",
    "bill_july",
    "bill_jun",
    "bill_may",
    "bill_apr"
]

positive_bills = df[bill_cols].clip(lower=0)

utilization = positive_bills.div(
    df["credit_limit"].replace(0, np.nan),
    axis=0
)

In [32]:
df["utilization_sep"] = utilization["bill_sep"]

df["avg_utilization"] = utilization.mean(axis=1)

df["max_utilization"] = utilization.max(axis=1)

In [33]:
df["avg_bill_6m"] = positive_bills.mean(axis=1)

df["avg_bill_utilization"] = (
    df["avg_bill_6m"] /
    df["credit_limit"].replace(0, np.nan)
)

In [34]:
# Handle any resulting missing values:
utilization_features = [
    "utilization_sep",
    "avg_utilization",
    "max_utilization",
    "avg_bill_utilization"
]

df[utilization_features] = (
    df[utilization_features]
    .fillna(0)
)

### Validation

The utilization features are checked for missing values and their
distribution is inspected for extreme observations.

In [35]:
df[utilization_features].isnull().sum()

utilization_sep         0
avg_utilization         0
max_utilization         0
avg_bill_utilization    0
dtype: int64

In [36]:
df[utilization_features].describe().T

,count,mean,std,min,25%,50%,75%,max
utilization_sep,29965.0,0.424430,0.411240,0.0,0.022300,0.315278,0.830325,6.455300
avg_utilization,29965.0,0.373806,0.351708,0.0,0.030531,0.285678,0.688706,5.364308
max_utilization,29965.0,0.495556,0.432953,0.0,0.071108,0.431600,0.923491,10.688575
avg_bill_utilization,29965.0,0.373806,0.351708,0.0,0.030531,0.285678,0.688706,5.364308


In [37]:
# Removing the redundant feature
df = df.drop(columns=["avg_bill_utilization"])

In [38]:
utilization_features = [
    "utilization_sep",
    "avg_utilization",
    "max_utilization"
]

In [39]:
df[utilization_features].describe().T

,count,mean,std,min,25%,50%,75%,max
utilization_sep,29965.0,0.424430,0.411240,0.0,0.022300,0.315278,0.830325,6.455300
avg_utilization,29965.0,0.373806,0.351708,0.0,0.030531,0.285678,0.688706,5.364308
max_utilization,29965.0,0.495556,0.432953,0.0,0.071108,0.431600,0.923491,10.688575


## 4. Payment Behavior Features

The six payment variables describe the customer's previous payment
amounts.

We create aggregate and trend features to summarize payment behavior:

- total payments over six months
- payments during the most recent three months
- payments during the earlier three months
- recent-to-older payment trend

Extreme payment-to-bill ratios are not included in the primary feature
set because their distributions are highly unstable when statement
amounts are very small.

In [40]:
payment_cols = [
    "payment_sep",
    "payment_aug",
    "payment_july",
    "payment_jun",
    "payment_may",
    "payment_apr"
]

In [41]:
df["total_payment_6m"] = df[payment_cols].sum(axis=1)

In [43]:
# Recent three months:
df["recent_payment_3m"] = df[
    ["payment_sep", "payment_aug", "payment_july"]
].sum(axis=1)

In [44]:
# Earlier three months:
df["older_payment_3m"] = df[
    ["payment_jun", "payment_may", "payment_apr"]
].sum(axis=1)

In [45]:
# The trend:
df["payment_trend_ratio"] = (
    df["recent_payment_3m"] /
    df["older_payment_3m"].replace(0, np.nan)
).fillna(0)

### Validation

The engineered payment features are checked for missing values,
infinite values, and distributional extremes.

In [46]:
payment_features = [
    "total_payment_6m",
    "recent_payment_3m",
    "older_payment_3m",
    "payment_trend_ratio"
]

In [47]:
df[payment_features].isnull().sum()

total_payment_6m       0
recent_payment_3m      0
older_payment_3m       0
payment_trend_ratio    0
dtype: int64

In [48]:
np.isinf(df[payment_features]).sum()

total_payment_6m       0
recent_payment_3m      0
older_payment_3m       0
payment_trend_ratio    0
dtype: int64

In [49]:
df[payment_features].describe().T

,count,mean,std,min,25%,50%,75%,max
total_payment_6m,29965.0,31687.783848,60853.841129,0.0,6700.000000,14400.000000,33600.000000,3764066.0
recent_payment_3m,29965.0,16829.771333,40994.238736,0.0,3646.000000,7054.000000,16420.000000,2978066.0
older_payment_3m,29965.0,14858.012515,32258.287631,0.0,2096.000000,5750.000000,14182.000000,786000.0
payment_trend_ratio,29965.0,4.548109,148.792735,0.0,0.518011,1.122247,1.875546,21000.0


In [50]:
df = df.drop(columns=["payment_trend_ratio"])

In [51]:
payment_features = [
    "total_payment_6m",
    "recent_payment_3m",
    "older_payment_3m"
]

## 5. Final Feature Validation

The final engineered features are checked for missing values,
infinite values, and basic statistical consistency before the
model-ready dataset is saved.

In [52]:
engineered_features = [
    "num_delayed_months",
    "max_delay",
    "avg_delay",
    "recent_delay",
    "utilization_sep",
    "avg_utilization",
    "max_utilization",
    "total_payment_6m",
    "recent_payment_3m",
    "older_payment_3m"
]

In [53]:
df[engineered_features].isnull().sum()

num_delayed_months    0
max_delay             0
avg_delay             0
recent_delay          0
utilization_sep       0
avg_utilization       0
max_utilization       0
total_payment_6m      0
recent_payment_3m     0
older_payment_3m      0
dtype: int64

In [54]:
np.isinf(df[engineered_features]).sum()

num_delayed_months    0
max_delay             0
avg_delay             0
recent_delay          0
utilization_sep       0
avg_utilization       0
max_utilization       0
total_payment_6m      0
recent_payment_3m     0
older_payment_3m      0
dtype: int64

In [55]:
df[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
num_delayed_months,29965.0,0.834273,1.554821,0.0,0.000000,0.000000,1.000000,6.000000e+00
max_delay,29965.0,0.682163,1.073847,0.0,0.000000,0.000000,2.000000,8.000000e+00
avg_delay,29965.0,0.621311,0.931530,0.0,0.000000,0.000000,1.666667,6.000000e+00
recent_delay,29965.0,0.356416,0.760752,0.0,0.000000,0.000000,0.000000,8.000000e+00
utilization_sep,29965.0,0.424430,0.411240,0.0,0.022300,0.315278,0.830325,6.455300e+00
avg_utilization,29965.0,0.373806,0.351708,0.0,0.030531,0.285678,0.688706,5.364308e+00
max_utilization,29965.0,0.495556,0.432953,0.0,0.071108,0.431600,0.923491,1.068858e+01
total_payment_6m,29965.0,31687.783848,60853.841129,0.0,6700.000000,14400.000000,33600.000000,3.764066e+06
recent_payment_3m,29965.0,16829.771333,40994.238736,0.0,3646.000000,7054.000000,16420.000000,2.978066e+06
older_payment_3m,29965.0,14858.012515,32258.287631,0.0,2096.000000,5750.000000,14182.000000,7.860000e+05


## 6. Save Feature-Engineered Dataset

The cleaned dataset is now supplemented with the validated financial
behavior features. The resulting dataset is saved separately so that
the modeling notebook can load it independently.

In [56]:
feature_path = "../data/processed/credit_risk_features.csv"

df.to_csv(feature_path, index=False)

print(f"Saved to: {feature_path}")
print(f"Shape: {df.shape}")

Saved to: ../data/processed/credit_risk_features.csv
Shape: (29965, 35)


In [57]:
original_cols = [
    "credit_limit", "gender", "education", "marital_status", "age",
    "repayment_sep", "repayment_aug", "repayment_july",
    "repayment_jun", "repayment_may", "repayment_apr",
    "bill_sep", "bill_aug", "bill_july", "bill_jun",
    "bill_may", "bill_apr",
    "payment_sep", "payment_aug", "payment_july",
    "payment_jun", "payment_may", "payment_apr",
    "default"
]

extra_cols = [col for col in df.columns if col not in original_cols]

print("Extra columns:")
print(extra_cols)
print("\nNumber of extra columns:", len(extra_cols))

Extra columns:
['num_delayed_months', 'max_delay', 'avg_delay', 'recent_delay', 'utilization_sep', 'avg_utilization', 'max_utilization', 'avg_bill_6m', 'total_payment_6m', 'recent_payment_3m', 'older_payment_3m']

Number of extra columns: 11


In [58]:
df.shape

(29965, 35)

In [59]:
df[engineered_features].shape

(29965, 10)

In [60]:
df = df.drop(columns=["avg_bill_6m"])
print("Shape:", df.shape)

Shape: (29965, 34)


In [61]:
print(engineered_features)
print("Number of engineered features:", len(engineered_features))

['num_delayed_months', 'max_delay', 'avg_delay', 'recent_delay', 'utilization_sep', 'avg_utilization', 'max_utilization', 'total_payment_6m', 'recent_payment_3m', 'older_payment_3m']
Number of engineered features: 10


In [62]:
feature_path = "../data/processed/credit_risk_features.csv"

df.to_csv(feature_path, index=False)

print(f"Saved to: {feature_path}")
print(f"Shape: {df.shape}")

Saved to: ../data/processed/credit_risk_features.csv
Shape: (29965, 34)


# Summary

The cleaned credit-card dataset was extended with ten financially
meaningful features capturing repayment behavior, credit utilization,
and payment activity.

The final feature-engineered dataset contains 29,965 observations and
34 columns, consisting of the original variables plus the engineered
features.

This dataset will be used as the input to the credit-risk modeling
stage.